# medqa-lora-vs-rag: Colab runner

Runs the full experiment on a Colab GPU with everything on Google Drive.

Before you start:

1. Runtime -> Change runtime type -> GPU (T4 is enough; A100 is faster).
2. Put the dataset CSV on Drive at `MyDrive/medqa-lora-vs-rag/data/disease_symptoms.csv`.
3. Optional: add a Colab secret named `WANDB_API_KEY` (key icon in the left bar) and enable notebook access.

If the session disconnects, re-run the cells from the top. Data prep, embeddings and trained adapters are
cached on Drive, and LoRA training resumes from its last checkpoint.

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists("medqa-lora-vs-rag"):
    !git clone -b develop https://github.com/nisha-s/medqa-lora-vs-rag.git
%cd medqa-lora-vs-rag
!git pull -q
!pip install -q -r requirements.txt
os.environ["PYTHONPATH"] = "src"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CSV = "/content/drive/MyDrive/medqa-lora-vs-rag/data/disease_symptoms.csv"
assert os.path.exists(CSV), f"put the dataset at {CSV} first"
print("dataset found:", os.path.getsize(CSV) // 1_000_000, "MB")

In [ ]:
# W&B key from Colab secrets; falls back to disabled logging so the run still works.
import os
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("W&B key loaded")
except Exception as err:
    os.environ["WANDB_MODE"] = "disabled"
    print("no WANDB_API_KEY secret, logging disabled:", err)

## Smoke test

A 1-step run on 50 rows. Takes a couple of minutes and catches path or install problems before the real run.

In [ ]:
!bash scripts/prepare_data.sh configs/colab_debug.yaml
!bash scripts/run_all.sh configs/colab_debug.yaml

## Full run

`run_all.sh` does everything below in order. The individual cells after it are the same steps, for re-running
one piece at a time. Finished runs are skipped, so re-running after a disconnect is safe.

In [ ]:
!bash scripts/prepare_data.sh configs/colab.yaml

In [ ]:
!bash scripts/run_all.sh configs/colab.yaml

### Individual steps

In [ ]:
# Retrieval-only control, both retrievers, every k
for retriever in ["jaccard", "embedding"]:
    for k in [1, 3, 5, 10]:
        !python -m medqa.evaluate --config configs/colab.yaml --arm retrieval_only --retriever {retriever} --k {k}

In [ ]:
# Base model arms
!python -m medqa.evaluate --config configs/colab.yaml --arm base_zero_shot
for k in [1, 3, 5, 10]:
    !python -m medqa.evaluate --config configs/colab.yaml --arm base_rag --k {k}

In [ ]:
# LoRA training and evaluation, one adapter per seed
for seed in [42, 43, 44]:
    !python -m medqa.train_lora --config configs/colab.yaml --seed {seed}
    !python -m medqa.evaluate --config configs/colab.yaml --arm lora --seed {seed}
    !python -m medqa.evaluate --config configs/colab.yaml --arm lora_rag --seed {seed}

In [ ]:
# LoRA + RAG over k, seed 42 only (the k ablation does not need all seeds)
for k in [1, 3, 10]:
    !python -m medqa.evaluate --config configs/colab.yaml --arm lora_rag --seed 42 --k {k}

In [ ]:
# MMLU retention for base and every trained adapter
!python -m medqa.evaluate --config configs/colab.yaml --mmlu

## Results

In [ ]:
!python -m medqa.evaluate --config configs/colab.yaml --aggregate
import pandas as pd
OUT = "/content/drive/MyDrive/medqa-lora-vs-rag/outputs"
display(pd.read_csv(f"{OUT}/results.csv"))
display(pd.read_csv(f"{OUT}/mmlu.csv"))

## Second model (optional)

Repeat the full run with Qwen2.5-1.5B-Instruct by writing a config that overrides the model name.

In [ ]:
config = '''extends: colab.yaml
output_dir: /content/drive/MyDrive/medqa-lora-vs-rag/outputs_qwen
model:
  name: Qwen/Qwen2.5-1.5B-Instruct
'''
open("configs/colab_qwen.yaml", "w").write(config)
# Splits are model-independent; copy them so the new output dir sees the same data.
!mkdir -p /content/drive/MyDrive/medqa-lora-vs-rag/outputs_qwen
!cp -r /content/drive/MyDrive/medqa-lora-vs-rag/outputs/splits /content/drive/MyDrive/medqa-lora-vs-rag/outputs_qwen/
!bash scripts/run_all.sh configs/colab_qwen.yaml